<a href="https://colab.research.google.com/github/ted-chang80/AIFFEL_quest_eng/blob/main/LLM_Application/LLM03/RAG_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

last modified date : 2026.03.15  
제작 : 박광석 (모두의연구소)

# 랭체인으로 RAG 시작하기

해당 노트는 Langchain으로 RAG를 구현하기 위해 필요한
각 컴포넌트인 Document Loaders, Text splitters, Text embeddings, Vectorstores, Retriever를 다룹니다  




### Step 0 : 설치와 준비  
Langchain 설치 및 Gemini API 키를 등록하도록 합니다.  

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install -U -q langchain langchain-openai
!pip install -U -q langchain-community langchain-core
!pip install -U langchain-text-splitters


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.3/114.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.3/235.3 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 9.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [19]:
import os


In [20]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_KEY')

In [6]:
#! curl ipinfo.io

In [21]:
from langchain_openai import ChatOpenAI

# OpenAI API를 사용하는 설정으로 변경
# 모델명은 필요에 따라 "gpt-4o", "gpt-4-turbo", "gpt-3.5-turbo" 등으로 바꿀 수 있습니다.
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.0,
)

In [22]:
!pip install -q pypdf pdf2image docx2txt pdfminer unstructured #의존성 모듈을 설치합니다

### Step 1 : Document Loaders 사용해보기  

Document Loader는 다양한 형태의 원본 데이터를  
LLM이 이해할 수 있는 Document 객체(text + metadata) 로 변환하는 역할을 합니다.

PDF, 웹페이지, CSV와 같이 형식이 서로 다른 문서들을 일관된 구조로 파싱하여, 이후 Chunking·Embedding·검색(Retrieval) 단계에서
바로 사용할 수 있도록 만들어줍니다.

즉, Document Loader는
**RAG 파이프라인의 가장 첫 단계에서 “데이터를 읽을 수 있는 형태로 정리하는 역할을 담당**합니다.

공식 문서에서는 지원되는 다양한 Loader 목록을 확인할 수 있습니다.
https://python.langchain.com/docs/modules/data_connection/document_loaders/

#### PDFLoader 사용  
이번 실습에서는 가장 많이 사용되는 문서 형식인 PDF 파일을 대상으로
PyPDFLoader를 사용해 문서를 불러옵니다.

실습을 위해, 질의응답에 활용하고 싶은 PDF 파일을 먼저 Colab 환경(또는 Drive)에 업로드해주세요.

PDFLoader는 각 페이지를 하나의 Document 단위로 변환하며,
이 단계에서 생성된 문서들은 이후 Text Splitter를 통해 의미 단위로 다시 분할됩니다.

In [23]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load_and_split()

In [24]:
pages[0]

Document(metadata={'producer': 'Adobe Acrobat Standard DC 19 Paper Capture Plug-in', 'creator': 'ScanFix(TM) Enhanced', 'creationdate': '2015-09-10T01:40:29+00:00', 'moddate': '2019-01-30T17:47:47+01:00', 'source': '/content/Demian.pdf', 'total_pages': 182, 'page': 0, 'page_label': '1'}, page_content='DEMIAN \n• \nDownloaded from https://www.holybooks.com')

In [25]:
print(pages[10])

page_content='TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut F

출력 결과를 보기 쉽게 확인하기 위해,
Document 객체 전체가 아닌 실제 텍스트 본문이 담긴 page_content만 선택하여 확인해보겠습니다.

In [26]:
print(pages[10].page_content)

TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut Franz Kromer ga

#### CSVLoader

SV 파일은 행(row) 단위로 구조화된 데이터를 담고 있는 형식으로,
LangChain의 CSVLoader를 사용하면 각 행을 하나의 Document 객체로 변환할 수 있습니다.

이렇게 변환된 문서들은 이후 PDF나 웹 문서와 동일하게
Embedding, VectorStore, Retrieval 단계에서 함께 활용할 수 있습니다.

실습을 위해, CSV 파일을 먼저 Colab 환경(또는 Drive)에 업로드해주세요.

In [27]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader("/content/titanic.csv")

data = loader.load()

In [28]:
data[:3]

[Document(metadata={'source': '/content/titanic.csv', 'row': 0}, page_content='PassengerId: 1\nSurvived: 0\nPclass: 3\nName: Braund, Mr. Owen Harris\nSex: male\nAge: 22\nSibSp: 1\nParch: 0\nTicket: A/5 21171\nFare: 7.25\nCabin: \nEmbarked: S'),
 Document(metadata={'source': '/content/titanic.csv', 'row': 1}, page_content='PassengerId: 2\nSurvived: 1\nPclass: 1\nName: Cumings, Mrs. John Bradley (Florence Briggs Thayer)\nSex: female\nAge: 38\nSibSp: 1\nParch: 0\nTicket: PC 17599\nFare: 71.2833\nCabin: C85\nEmbarked: C'),
 Document(metadata={'source': '/content/titanic.csv', 'row': 2}, page_content='PassengerId: 3\nSurvived: 1\nPclass: 3\nName: Heikkinen, Miss. Laina\nSex: female\nAge: 26\nSibSp: 0\nParch: 0\nTicket: STON/O2. 3101282\nFare: 7.925\nCabin: \nEmbarked: S')]

#### 웹베이스로더  
웹베이스 로더는 웹페이지에 포함된 텍스트 콘텐츠를 직접 파싱하여 Document 객체로 변환하는 역할을 합니다.  
이를 통해 뉴스 기사, 블로그 글, 공지사항과 같은 실시간으로 업데이트되는 웹 문서를 RAG 시스템의 지식 소스로 활용할 수 있습니다.  
이번 실습에서는 실제 뉴스 기사를 예제로 사용하여,
웹페이지의 내용을 불러오고 텍스트 형태로 변환하는 과정을 살펴봅니다.  

실습에 사용할 웹페이지는 다음과 같습니다.  
https://it.chosun.com/news/articleView.html?idxno=2023092111831

In [29]:
from langchain_community.document_loaders import WebBaseLoader

In [30]:
loader = WebBaseLoader("https://it.chosun.com/news/articleView.html?idxno=2023092111831")
documents = loader.load()

#print(documents[0].page_content)

주석을 해제하고 코드를 실행하면,
해당 웹페이지에 포함된 본문 텍스트 전체를 불러와 확인할 수 있습니다.  

웹페이지, PDF, CSV 등 서로 다른 형식의 문서들이
모두 텍스트 형태로 정상적으로 파싱된 것을 확인할 수 있습니다.  

이제 이 텍스트를 **전처리(불필요한 요소 제거, 정제)** 한 뒤,
Chunking과 Embedding 단계에 활용할 수 있습니다.  

### Step2 : TextSplitters 사용해보기  
Text Splitter는 긴 텍스트 문서를 **의미를 유지한 작은 단위(Chunk)** 로 분할하는 역할을 합니다.  
LLM은 한 번에 처리할 수 있는 토큰 수에 제한이 있기 때문에, 문서를 그대로 입력하는 대신 Splitter를 통해 분할된 여러 Chunk를 입력받아 처리하게 됩니다.  

이 과정을 통해 긴 문서에서도 토큰 길이 제약을 극복하고, 필요한 부분만 효율적으로 검색할 수 있습니다.  

분할된 각 Chunk는 이후 단계에서 1:1로 Embedding되어 VectorStore에 저장되며,
이 Chunk 단위가 RAG 시스템에서 검색과 응답의 기본 단위가 됩니다.  

In [31]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

CharacterTextSplitter는
하나의 고정된 구분자(separator)를 기준으로 텍스트를 분할하는 방식입니다.
구현이 단순하고 직관적이지만,
문서 구조에 따라 분할된 Chunk가 토큰 제한을 초과하는 경우가 발생할 수 있습니다.

반면, RecursiveCharacterTextSplitter는
줄바꿈, 문장 구분자, 구두점 등 여러 구분자를 순차적으로 적용하며
텍스트를 재귀적으로 분할합니다.

이 방식은 토큰 제한을 안정적으로 만족시키는 데 유리하지만,
분할 과정에서 의미적으로 완전하지 않은 문장 단위로 잘릴 수 있다는 단점이 있습니다.  

단순한 구조의 문서나,
문단 구성이 명확한 텍스트의 경우에는 CharacterTextSplitter로도 충분합니다.

하지만 실제 서비스 환경에서는
문서 길이와 구조가 제각각인 경우가 많기 때문에,
대부분의 RAG 시스템에서는 RecursiveCharacterTextSplitter를 기본 선택지로 사용합니다.

이는 Chunk 크기를 안정적으로 제어하면서도
검색 실패를 줄이는 데 유리하기 때문입니다.

In [32]:
with open("/content/state_of_the_union.txt") as f:
    text = f.read()

In [33]:
#len은 어떤 기준으로 chunk size를 잴 것인가?의 기준이 되는 함수입니다.
#chunk_overlap은 chunk의 앞뒤로 다른 chunk와 설정한 size까지 겹칠 수 있도록 설정하는 것입니다.
text_splitter = CharacterTextSplitter(separator="\n\n", chunk_size=1000, chunk_overlap=100, length_function = len,)
chunks = text_splitter.split_text(text)

Chunk의 내용을 확인해보겠습니다

In [34]:
print(chunks[0])

Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  

Last year COVID-19 kept us apart. This year we are finally together again. 

Tonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. 

With a duty to one another to the American people to the Constitution. 

And with an unwavering resolve that freedom will always triumph over tyranny. 

Six days ago, Russia’s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. 

He thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. 

He met the Ukrainian people. 

From President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determination, inspires the world.


각 chunk의 길이를 확인해보겠습니다,

In [35]:
length = []
for chunk in chunks:
    length.append(len(chunk))

print(length)

[939, 995, 810, 962, 994, 883, 957, 952, 928, 915, 993, 702, 900, 950, 957, 958, 967, 996, 796, 866, 888, 966, 964, 977, 998, 948, 925, 924, 989, 965, 938, 936, 981, 965, 771, 982, 972, 977, 984, 999, 968, 801]


### 토큰 단위로 텍스트 분할해보기  
  
LLM은 문장을 단어가 아닌 토큰(token) 단위로 처리합니다.
따라서 사람이 인식하는 단어 길이나 문자 수는
실제 모델이 처리하는 입력 길이와 정확히 일치하지 않을 수 있습니다.

이로 인해 문자 수나 단어 수를 기준으로 텍스트를 분할할 경우,
모델의 입력 토큰 제한을 초과하거나
예상보다 훨씬 짧은 문맥만 전달되는 문제가 발생할 수 있습니다.

실제 서비스 환경에서는 이러한 문제를 방지하기 위해,
토큰 단위를 기준으로 텍스트를 분할하는 방식을 사용합니다.
이제 토큰 기준으로 텍스트를 분할해보겠습니다.

In [36]:
!pip install tiktoken

In [37]:
import tiktoken
tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    tokens = tokenizer.encode(
        text
    )
    return len(tokens)

In [38]:
tiktoken_length = []
for chunk in chunks:
    tiktoken_length.append(tiktoken_len(chunk))

print(length)
print(tiktoken_length)

[939, 995, 810, 962, 994, 883, 957, 952, 928, 915, 993, 702, 900, 950, 957, 958, 967, 996, 796, 866, 888, 966, 964, 977, 998, 948, 925, 924, 989, 965, 938, 936, 981, 965, 771, 982, 972, 977, 984, 999, 968, 801]
[197, 198, 163, 190, 203, 182, 195, 197, 206, 205, 218, 148, 188, 205, 216, 215, 209, 224, 176, 187, 201, 197, 201, 215, 222, 202, 203, 204, 229, 206, 184, 204, 197, 194, 156, 200, 194, 221, 203, 225, 209, 187]


글자 수와 토큰 수의 차이를 확인할 수 있습니다 !

### Step3 : TextEmbedding 사용해보기  
Embedding은 텍스트를 컴퓨터가 계산할 수 있는 수치 벡터(vector) 형태로 변환하는 과정입니다.
이 벡터는 문장의 표면적인 형태가 아니라, 의미적 유사성을 반영하도록 설계되어 있습니다.

변환된 벡터는
VectorStore에 저장되거나,
새로운 질의(Query) 벡터와의 유사도 계산을 통해
의미적으로 가까운 문서를 검색하는 데 사용됩니다.

이러한 변환은 대규모 말뭉치로 사전 학습된
Embedding 전용 모델을 통해 이루어지며,
RAG 시스템에서 Retrieval 성능을 결정하는 핵심 요소입니다.

이번 실습에서는
OpenAI 임베딩 모델을 사용해
텍스트를 벡터로 변환해보겠습니다.

In [39]:
import openai

genai 라이브러리의 list_models 함수를 사용하여 사용 가능한 모델들의 목록을 가져옵니다.

In [40]:
client = openai.OpenAI()

In [41]:
for model in client.models.list():
    if "embedding" in model.id:
        print(model.id)

text-embedding-ada-002
text-embedding-3-small
text-embedding-3-large


text-embedding-3-small은 가성비가 좋고, text-embedding-3-large는 성능이 더 강력합니다.

In [42]:
from langchain_openai import OpenAIEmbeddings


embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

만약 여러분들이 Gemini를 사용하여 구축중이시라면, embedding은 지역에 따라 사용이 제한됩니다.  
주로 유럽권에서 제한되기 때문에, 다음 에러를 확인하신다면 Colab 파일의 서버 저장 위치를 확인 후, 다른 임베딩 모델로 변경해야합니다.  

Error embedding content: 400 User location is not supported for the API use.


In [43]:
#!curl ipinfo.io

In [44]:
# 400 User location is not supported for the API use 오류가 발생한다면, 이 블록을 대신 실행해주세요

# ! pip install -q sentence_transformers

#from langchain.embeddings import HuggingFaceEmbeddings
#embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

embedding model 변수에 OpenAI 임베딩모델 혹은 huggingface의 임베딩모델이 할당되었을 것입니다.  
embed_documents 멤버 함수를 사용하여 새 문장을 변환해보겠습니다  

In [45]:
embeddings = embedding_model.embed_documents(
    [
        "This is red apple.",
        "This is yellow banana.",
        "This is green lime.",
    ]
)

임베딩으로 잘 변환되었는지 확인해보겠습니다  

In [46]:
print(embeddings[1])

[0.006435394287109375, -0.0208587646484375, -0.0216064453125, 0.0235595703125, -0.0428466796875, -0.004608154296875, 0.03973388671875, 0.05267333984375, -0.0018892288208007812, -0.0237579345703125, 0.0137939453125, 0.0159912109375, -0.044464111328125, -0.00013935565948486328, -0.007648468017578125, 0.02557373046875, 0.0254058837890625, 0.035491943359375, -0.0517578125, 0.01232147216796875, 0.0248260498046875, -0.00045800209045410156, 0.0193939208984375, 0.07525634765625, -0.0020503997802734375, -0.00921630859375, 0.016815185546875, -0.007152557373046875, 0.02655029296875, -0.048492431640625, 0.043182373046875, -0.04254150390625, 0.01165008544921875, -0.032867431640625, -0.0333251953125, -0.046142578125, 0.0019063949584960938, 0.0225677490234375, -0.01812744140625, 0.03253173828125, 0.0293121337890625, 0.011749267578125, -0.01442718505859375, 0.0030574798583984375, 0.0243377685546875, 0.048187255859375, -0.01355743408203125, 0.049346923828125, -0.040618896484375, 0.04400634765625, 0.045

In [47]:
len(embeddings[1])

1536

새로운 쿼리를 넣어, 임베딩끼리 유사도를 계산해보겠습니다

In [48]:
import numpy as np
from numpy import dot
from numpy.linalg import norm
def cos_sim(A, B):
  return dot(A, B)/(norm(A)*norm(B))

In [49]:
query = ["this is red fruit"]

In [50]:
e_query = embedding_model.embed_documents(query)
print(cos_sim(embeddings[0], e_query[0]))
print(cos_sim(embeddings[1], e_query[0]))
print(cos_sim(embeddings[2], e_query[0]))

0.7477942151308524
0.48995458116791124
0.4084112226829356


빨간 사과와 빨간 과일의 유사도가 많이 높게 나왔습니다!  
  
임베딩 모델은 사용 언어나 필요에 따라 다양하게 교체하여 사용할 수 있습니다.  
해당 링크에서 여러 목록을 확인하실 수 있습니다.  
https://python.langchain.com/docs/integrations/text_embedding/

### Step4 : VectorStore 사용해보기
VectorStore는 텍스트를 Embedding 모델을 통해 벡터(vector)로 변환한 뒤, 이를 저장하고 관리하는 저장소입니다.
이 저장소는 단순한 데이터 보관 공간이 아니라,
벡터 간의 유사도를 빠르게 계산하고 탐색하기 위한 인덱싱 구조를 함께 포함하고 있습니다.

문서나 쿼리가 Embedding된 이후에는,
VectorStore를 통해 의미적으로 유사한 벡터를 효율적으로 검색할 수 있으며,
이 과정이 RAG 시스템의 Retrieval 단계를 담당하게 됩니다.

대표적인 VectorStore로는
Chroma, FAISS 등이 있으며,
각각 로컬 환경과 대규모 서비스 환경에서 널리 사용됩니다.

이번 실습에서는
구성이 단순하고 로컬 환경에서 바로 사용할 수 있는
ChromaDB를 사용해 VectorStore를 구성해보겠습니다.

In [51]:
!pip install chromadb

In [52]:
!pip install langchain-chroma

In [53]:
from langchain_chroma import Chroma

In [54]:
!pip install --upgrade opentelemetry-api
!pip install --upgrade opentelemetry-sdk

제일 처음에 사용했던, PDF를 다시 사용하도록 합니다!  

In [55]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken

tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    tokens = tokenizer.encode(
        text
    )
    return len(tokens)

# 위에서 사용했던 코드입니다
loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load_and_split()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, length_function = tiktoken_len)
docs = text_splitter.split_documents(pages)

In [56]:
#!pip show chromadb

Chroma에 임베딩 시킵니다  

In [57]:
db = Chroma.from_documents(docs, embedding_model)


In [58]:
from langchain_openai import ChatOpenAI
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from google.colab import userdata

# OpenAI 모델로 변경
# streaming=True와 callbacks 설정을 통해 실시간 출력을 활성화합니다.
llm = ChatOpenAI(
    model="gpt-4o",              # 또는 "gpt-4o-mini"
    temperature=0.0,
    streaming=True,              # 실시간 출력을 켭니다
    callbacks=[StreamingStdOutCallbackHandler()], # 출력을 콘솔에 바로 뿌려줍니다
    api_key=userdata.get('OPENAI_KEY') # API 키를 직접 전달
)

이제 쿼리를 날려보겠습니다

In [59]:
query = "how Demian look like?"
docs = db.similarity_search(query)

In [60]:
print(docs[0].page_content)

DEMIAN 
with a feeling of nausea, I noticed Demian's expression. 
He had not thrust himself to the front but stood right 
at the back, looking elc!gant and at ease as usual. His 
glance seemed directed at the horse's head, and again it 
. showed that deep, quiet, almost fanatical yet passionate 
absorption. I could not help staring at him for some 
moments and it was then that I felt aware of a very 
uncanny sensation in my remote consciousness. I saw 
Demian's face and remarked that it was not a boy's face 
but a man's and then I saw, or rather became aware, that 
it was not really the face of a man either; it had some­
thing different about it, almost a feminine element. And 
for the time being his face seemed neither masculine 
nor childish, neither old nor young but a hundred years 
old, almost timeless and bearing the mark of other 
periods of history than our own. Animals might look 
thus, trees or stars. I did not know then, of course, I 
did not feel exactly what I am writing a

Face, features, looks like 등 데미안의 생김새를 담고 있는 페이지가 출력되었습니다  
굉장히 빠른 속도로 검색했습니다!  

### Step5 : Retriever 사용해보기  

Retriever는 사용자의 질문을 Embedding 모델을 통해 벡터로 변환한 뒤,
VectorStore에 저장된 문서 벡터들과 비교하여
의미적으로 가장 유사한 문서(Chunk)를 찾아 반환하는 역할을 합니다.

즉, Retriever는
RAG 시스템에서 “어떤 정보를 LLM에게 참고 자료로 줄 것인가”를 결정하는 핵심 컴포넌트이며,
검색 결과의 품질이 곧 최종 답변의 품질로 이어집니다.
  

In [61]:
!pip install -U langchain langchain-classic

In [62]:
from langchain_classic.chains.retrieval_qa.base import RetrievalQA


긴 문서 전체를 한 번에 LLM에 전달하는 대신,
Retriever와 LLM을 결합한 RetrievalQA 체인을 사용하여
문서에서 질문과 관련된 부분만 검색하고,
그 결과를 바탕으로 답변을 생성합니다.

이를 통해 길이가 긴 문서에서도
토큰 제한을 넘지 않으면서, 근거 기반의 질의응답을 수행할 수 있습니다.

In [69]:
from langchain_openai import ChatOpenAI
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# OpenAI 모델로 변경
# streaming=True와 callbacks 설정을 통해 실시간 출력을 활성화합니다.
llm = ChatOpenAI(
    model="gpt-5-nano",              # 또는 "gpt-4o-mini"
    temperature=0.0,
    streaming=True,              # 실시간 출력을 켭니다
    callbacks=[StreamingStdOutCallbackHandler()], # 출력을 콘솔에 바로 뿌려줍니다
)

체인의 종류와 검색(Retrieval) 방식,
그리고 그에 따른 주요 파라미터를 설정합니다.

이 단계에서는
Retriever가 어떤 전략으로 문서를 검색할지,
그리고 몇 개의 문서를 LLM에게 전달할지를 결정하게 됩니다.
이 선택은 최종 답변의 품질과 직접적으로 연결됩니다.

예를 들어,
MMR(Maximal Marginal Relevance) 방식은
쿼리와의 유사도뿐만 아니라 문서 간의 중복을 줄이고 다양성을 확보하는 재정렬(Re-ranking) 전략입니다.

실무 환경에서는 단일 문서에 정보가 몰리는 것을 방지하고,
LLM이 보다 풍부한 문맥을 참고하도록 하기 위해
MMR 방식이 자주 사용됩니다.

In [70]:
from langchain_classic.chains.retrieval_qa.base import RetrievalQA
qa = RetrievalQA.from_chain_type(llm, chain_type="stuff",
                                 retriever=db.as_retriever(
                                     search_type="mmr",
                                     search_kwargs={"k": 3, "fetch_k" : 10}),
                                 return_source_documents=True)

위 코드에서 짚고 넘어갈 파라미터는 다음과 같습니다  
🔹 chain_type="stuff"

검색된 문서(Chunk)를 그대로 하나의 Prompt에 모두 삽입하는 방식입니다.
구조가 단순하고 이해하기 쉬워,
RAG 구조를 처음 학습하거나 프로토타입을 만들 때 적합합니다.
단점으로는 문서 수가 많아질 경우
토큰 사용량이 빠르게 증가할 수 있습니다.
실무에서는 초기 검증 단계에서는 stuff를,
문서 수가 많아지면 map_reduce나 refine 방식으로 확장합니다.  

🔹 retriever

VectorStore에서 어떤 문서를 검색할지 결정하는 검색 모듈입니다.
검색 전략과 파라미터 설정에 따라 LLM이 참고하는 정보의 범위와 품질이 달라집니다.  

🔹 search_type="mmr"

MMR(Maximal Marginal Relevance) 검색 방식을 사용합니다. 쿼리와의 유사도뿐만 아니라, 문서 간 중복을 줄여 다양한 문맥을 확보하는 Re-ranking 전략입니다. 실무 환경에서 단일 문서 편향을 줄이기 위해 자주 사용됩니다

🔹 search_kwargs={"k": 3, "fetch_k": 10}  
- fetch_k  
VectorStore에서 우선적으로 가져올 후보 문서 개수입니다. Re-ranking 이전 단계에서 사용됩니다.
- k  
최종적으로 LLM에게 전달할 문서(Chunk)의 개수입니다.

일반적으로 fetch_k > k 로 설정하여 후보 풀을 넉넉히 확보한 뒤, 품질 좋은 문서만 선별하는 방식을 사용합니다.  

🔹 return_source_documents=True

답변 생성에 사용된 원문 문서(Chunk)를 함께 반환합니다. 이를 통해 답변의 출처를 사용자에게 표시하거나 검색 품질을 디버깅하고 RAG 성능을 평가할 수 있습니다. 실무 서비스에서는 거의 필수적으로 사용하는 옵션입니다.

In [71]:
query = "how demian looks like"
result = qa(query)

In the text, Demian is described as elegant and composed, often standing back rather than pushing to the front. His face is not clearly a typical boy’s or man’s; it carries a feminine element and feels almost timeless—like something from another era. The narrator calls him “unimaginably different” and says his appearance is difficult to pin down as male or childish, sometimes seeming as old as a hundred years. He is described as handsome but also capable of repelling, and as if he were more like an animal, a spirit, or an image than a ordinary person.

There’s also a later image of his face (or Beatrice/Demian’s) in a painting by lamp-light: the outlines blur, but the eyes glow pink around them, the forehead shines, and the mouth is a striking red, suggesting a face that expresses the narrator’s inner fate.

If you’d like, I can quote a short passage or summarize a specific part in more detail.

마크다운 형식으로 출력해봅니다

In [72]:
from IPython.display import Markdown, display
display(Markdown(result["result"]))

In the text, Demian is described as elegant and composed, often standing back rather than pushing to the front. His face is not clearly a typical boy’s or man’s; it carries a feminine element and feels almost timeless—like something from another era. The narrator calls him “unimaginably different” and says his appearance is difficult to pin down as male or childish, sometimes seeming as old as a hundred years. He is described as handsome but also capable of repelling, and as if he were more like an animal, a spirit, or an image than a ordinary person.

There’s also a later image of his face (or Beatrice/Demian’s) in a painting by lamp-light: the outlines blur, but the eyes glow pink around them, the forehead shines, and the mouth is a striking red, suggesting a face that expresses the narrator’s inner fate.

If you’d like, I can quote a short passage or summarize a specific part in more detail.

RAG를 사용하지 않은 llm 호출도 시도해보세요!

In [67]:
llm2 = ChatOpenAI(
    model="gpt-4o")
request = llm2.invoke("how demian looks like")
display(Markdown(request.content))


If you’re referring to the character Demian from Hermann Hesse's novel "Demian," the book does not provide an exhaustive physical description, leaving much to the reader’s imagination. Demian is often depicted as enigmatic and charismatic, with a presence that conveys both maturity and innocence. He is portrayed as having a profound and intense gaze, often interpreted as piercing or knowing, which reflects his deep understanding of the world and spiritual awareness. This abstract description allows readers to imagine Demian in a way that aligns with their interpretation of his complex character. If you were thinking of a different "Demian," please provide more context.

### Quiz
결과의 어떤 부분을 관찰하였을 때, RAG 시스템의 결과를 신뢰할 수 있겠다 생각하셨나요?  

### Answer  
원문에서 답변의 출처를 확인할 수 있었습니다.

## 6. 완성 예제  
앞에서 진행한 내용으로, Demian을 다시 한번 읽어봅시다!  
완성하여 제출해주세요~


필요한 라이브러리를 모두 다운받습니다  

Text splitter 사용을 위한 준비입니다

### Step 1 Document loader

In [73]:
loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load_and_split()

### Step 2 Text splitters

In [74]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50, length_function=tiktoken_len)
docs = text_splitter.split_documents(pages)

### Step 3 Vector Empeddings

In [75]:
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small", api_key=userdata.get('OPENAI_KEY'))

In [76]:
db = Chroma.from_documents(docs, embedding_model)

### Step 4 Retrievers

In [77]:
retriever = db.as_retriever(search_type="mmr", search_kwargs={"k": 3})

In [78]:
qa = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever, return_source_documents=True)

### Step 5 Question Answering

In [79]:
query = "Who is Max Demian?"
result = qa.invoke(query)
display(Markdown(result["result"]))

Max Demian is a fictional character from Hermann Hesse’s novel Demian (Emil Sinclair’s Youth). He is Sinclair’s enigmatic classmate/friend who becomes a kind of mentor and inner guide, challenging Sinclair to think for himself and to question conventional morality.

Key points:
- He comes from a wealthy family; rumors surround him (nonconformist, not churchgoing; rumored to be talented and physically capable).
- He seems to know more than others and has a profound, unsettling presence.
- He introduces and embodies the idea that good and evil aren’t strictly separated and that each person must decide for themselves what is right or forbidden.
- In the narrative, Demian often appears as both a real friend and a symbolic figure—the “demon” or inner voice guiding Sinclair toward self-discovery and individuation.

If you’d like, I can give a brief quote or a specific scene that shows his influence.

Max Demian is a fictional character from Hermann Hesse’s novel Demian (Emil Sinclair’s Youth). He is Sinclair’s enigmatic classmate/friend who becomes a kind of mentor and inner guide, challenging Sinclair to think for himself and to question conventional morality.

Key points:
- He comes from a wealthy family; rumors surround him (nonconformist, not churchgoing; rumored to be talented and physically capable).
- He seems to know more than others and has a profound, unsettling presence.
- He introduces and embodies the idea that good and evil aren’t strictly separated and that each person must decide for themselves what is right or forbidden.
- In the narrative, Demian often appears as both a real friend and a symbolic figure—the “demon” or inner voice guiding Sinclair toward self-discovery and individuation.

If you’d like, I can give a brief quote or a specific scene that shows his influence.

In [81]:
from langchain_community.document_loaders import PyPDFLoader

# 1. 문서 로드 (실제 경로로 수정)
file_path = "/content/90일만에당신의회사를고수익기업으로바꿔라.pdf"
loader = PyPDFLoader(file_path)
documents = loader.load()

print(f"총 {len(documents)} 페이지의 문서를 불러왔습니다.")

총 264 페이지의 문서를 불러왔습니다.


In [101]:
# 0. OCR을 위한 필수 라이브러리 설치
!apt-get install -y -q poppler-utils
!pip install -q pdf2image easyocr

import os
from pdf2image import convert_from_path
import easyocr
import numpy as np
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_classic.chains.retrieval_qa.base import RetrievalQA
from langchain_text_splitters import RecursiveCharacterTextSplitter
from google.colab import userdata
import tiktoken

# 1. PDF를 이미지로 변환 후 OCR 수행
file_path = "/content/90일만에당신의회사를고수익기업" + "으로바꿔라.pdf"
reader = easyocr.Reader(['ko', 'en']) # 한국어와 영어 인식 설정
extracted_docs = []

try:
    print("PDF를 이미지로 변환 중... (페이지가 많아 시간이 소요될 수 있습니다)")
    # 메모리 관리를 위해 우선 앞부분 10페이지만 테스트로 처리해보거나 전체를 처리합니다.
    # 10페이지 테스트 후 전체 페이지를 처리합니다.
    # 여기서는 전체 흐름을 위해 우선 시도합니다.
    images = convert_from_path(file_path, first_page=1, last_page=264) # 테스트를 위해 20페이지만 우선 처리 후 전체 페이지 처리


    print(f"변환된 {len(images)}개 페이지에 대해 OCR 수행 중...")
    for i, image in enumerate(images):
        text_list = reader.readtext(np.array(image), detail=0)
        text = " ".join(text_list)
        if text.strip():
            extracted_docs.append(Document(page_content=text, metadata={"page": i + 1, "source": "OCR"}))

    print(f"성공적으로 {len(extracted_docs)}페이지의 텍스트를 추출했습니다.")
except Exception as e:
    print(f"OCR 처리 중 오류 발생: {e}")

# 2. RAG 파이프라인 구성
if extracted_docs:
    tokenizer = tiktoken.get_encoding("cl100k_base")
    def tiktoken_len(text):
        return len(tokenizer.encode(text))

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100, length_function=tiktoken_len)
    docs = text_splitter.split_documents(extracted_docs)

    embedding_model = OpenAIEmbeddings(model="text-embedding-3-small", api_key=userdata.get('OPENAI_KEY'))
    db_new = Chroma.from_documents(docs, embedding_model)

    qa_new = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=db_new.as_retriever(search_type="mmr", search_kwargs={"k": 3}),
        return_source_documents=True
    )
    print(f"성공적으로 {len(docs)}개의 청크를 생성했습니다. 이제 질문해 주세요!")

Reading package lists...
Building dependency tree...
Reading state information...
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 51 not upgraded.
PDF를 이미지로 변환 중... (페이지가 많아 시간이 소요될 수 있습니다)
변환된 264개 페이지에 대해 OCR 수행 중...
성공적으로 254페이지의 텍스트를 추출했습니다.
성공적으로 301개의 청크를 생성했습니다. 이제 질문해 주세요!


In [163]:
# image는 단일 객체이므로 인덱스로 접근할 수 없습니다.
# 전체 이미지 리스트인 images를 사용해야 합니다.
print(images[10])

<PIL.PpmImagePlugin.PpmImageFile image mode=RGB size=1166x1755 at 0x7A20906B0B00>


In [104]:
from IPython.display import Markdown, display

# 마지막 페이지까지 더 잘 검색되도록 파라미터 상향 조정
query = "이 책의 마지막 부분에서 강조하는 결론과 고수익 기업으로의 최종 전환을 위한 핵심 조언은 무엇인가요?"

if 'qa_new' in globals():
    # 검색 결과 개수(k)를 늘려 더 넓은 문맥을 참조하도록 retriever 설정 변경
    qa_new.retriever.search_kwargs = {"k": 10, "fetch_k": 30}

    print("🚀 문서 후반부를 포함하여 정밀 검색을 진행합니다...")
    result = qa_new.invoke(query)

    print(f"질문: {result['query']}")
    print("-" * 50)
    display(Markdown(result['result']))

    print("\n[참조 페이지 - 답변의 근거가 된 페이지들]")
    # 중복 제거 및 정렬하여 출력
    unique_pages = sorted(list(set([doc.metadata.get('page') for doc in result['source_documents']])))
    for p in unique_pages:
        print(f"페이지: {p}")
else:
    print("qa_new 체인이 생성되지 않았습니다. OCR 단계(B7RY6lbPM0JS)를 먼저 완료해 주세요.")

🚀 문서 후반부를 포함하여 정밀 검색을 진행합니다...
질문: 이 책의 마지막 부분에서 강조하는 결론과 고수익 기업으로의 최종 전환을 위한 핵심 조언은 무엇인가요?
--------------------------------------------------


이 책의 마지막 부분에서는 고수익 기업으로의 전환을 위해 감성 마케팅(이모셔널 마케팅)의 중요성을 강조합니다. 핵심 조언으로는 고객의 구매 심리에 따른 설계도를 만들고, 예상 고객을 비용 효과적으로 모아 성약하고 반복 구매가 이루어지도록 하는 것이 중요하다고 언급합니다. 또한, 광고에는 돈을 버는 광고와 그렇지 않은 광고가 있으며, 효과적인 광고를 통해 고객을 모으는 활동이 모든 것의 원점이라고 설명합니다.


[참조 페이지 - 답변의 근거가 된 페이지들]
페이지: 13
페이지: 114
페이지: 147
페이지: 195
페이지: 205
페이지: 206
페이지: 220
페이지: 239
페이지: 247
페이지: 250


In [109]:
# 0. 환경에 맞는 라이브러리 임포트 (langchain_classic 활용)
from langchain_openai import ChatOpenAI
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# 1. LLM (GPT-4o) 초기화
llm = ChatOpenAI(model_name="gpt-4o", temperature=0)

# 2. 검색기(Retriever) 설정
if 'db_new' in globals():
    retriever = db_new.as_retriever(search_type="mmr", search_kwargs={"k": 3})
    print("신규 PDF 문서 검색기를 사용합니다.")
else:
    retriever = db.as_retriever(search_type="mmr", search_kwargs={"k": 3})
    print("데미안 문서 검색기를 사용합니다.")

# 3. 프롬프트 템플릿 생성
system_prompt = (
    "당신은 주어진 문서에 기반하여 질문에 답변하는 유능한 어시스턴트입니다. "
    "반드시 아래에 제공된 검색된 문맥(context)만을 사용하여 질문에 답하세요. "
    "만약 제공된 문맥에서 답을 찾을 수 없다면, 모른다고 말하고 정보를 지어내지 마세요. "
    "답변은 한국어로 최대한 명확하고 상세하게 작성해주세요.\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# 4. RAG 체인 생성
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

# 5. 실행
query = "이 책에서 제시하는 고수익 기업으로 전환하기 위한 90일 계획의 핵심은 무엇인가요?"

try:
    result = rag_chain.invoke({"input": query})
    print("\n🤔 질문:", query)
    print("\n💡 GPT-4o의 답변:\n", result["answer"])
    print("\n📚 참조한 문서 조각 수:", len(result["context"]))
except Exception as e:
    print(f"\n❌ 오류 발생: {e}")

신규 PDF 문서 검색기를 사용합니다.

🤔 질문: 이 책에서 제시하는 고수익 기업으로 전환하기 위한 90일 계획의 핵심은 무엇인가요?

💡 GPT-4o의 답변:
 이 책에서 제시하는 고수익 기업으로 전환하기 위한 90일 계획의 핵심은 "이모셔널 마케팅(감성 마케팅)"을 활용하는 것입니다. 이 방법을 통해 실제로는 약 30일 후에 회사의 예금 잔액이 늘어날 수 있다고 주장합니다. 90일이라는 기간은 여러 가지를 고려하는 시간을 포함한 것이며, 실천을 시작하면 더 빠르게 결과를 볼 수 있다고 설명합니다.

📚 참조한 문서 조각 수: 3


In [110]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from IPython.display import Markdown, display

# 1. 답변의 구체성을 극대화하기 위한 전문 프롬프트 설정
system_prompt_detailed = (
    "당신은 '90일만에 고수익 기업으로 바꾸는 법'의 저자이자 전략 경영 컨설턴트입니다. "
    "검색된 문맥(context)을 바탕으로 매우 구체적이고 실행 가능한 답변을 작성하세요.\n\n"
    "답변 구성 가이드:\n"
    "1. 핵심 개념: 관련 이론이나 용어(예: 이모셔널 마케팅, 고객 획득 비용 등)를 명확히 정의하세요.\n"
    "2. 단계별 실행 방안: 90일 계획의 흐름에 맞춰 구체적인 액션 플랜을 제시하세요.\n"
    "3. 실전 팁: 문서에서 강조하는 사소하지만 중요한 디테일(광고 문구, 고객 응대법 등)을 포함하세요.\n"
    "4. 페이지 참조: 각 설명 끝에 [p.XX] 형태로 출처를 표시하세요.\n\n"
    "문맥에 없는 내용은 추측하지 말고, 정보가 부족하다면 '문서에 해당 내용이 명시되지 않았음'을 알리세요.\n\n"
    "Context: {context}"
)

# 2. 질문 재구성 및 체인 설정
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt_detailed),
    ("human", "{input}"),
])

# 3. 고급 RAG 체인 생성
# k=15로 설정하여 검색량을 대폭 늘리고, fetch_k를 통해 후보군을 충분히 확보합니다.
retriever_pro = db_new.as_retriever(search_type="mmr", search_kwargs={"k": 15, "fetch_k": 40})
question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
rag_chain_pro = create_retrieval_chain(retriever_pro, question_answer_chain)

# 4. 구체적인 분석 실행
query_advanced = "이 책에서 말하는 고수익 기업으로 가기 위한 90일 로드맵을 주차별 혹은 단계별로 매우 상세하게 정리해주고, 각 단계에서 사장이 반드시 해야 할 행동을 알려줘."

print("🔍 전체 문서를 정밀하게 스캔하여 구체적인 로드맵을 작성 중입니다...")
try:
    response = rag_chain_pro.invoke({"input": query_advanced})
    display(Markdown(f"## 🚀 고수익 기업 전환 정밀 컨설팅 보고서\n\n{response['answer']}"))
except Exception as e:
    print(f"실행 중 오류 발생: {e}")

🔍 전체 문서를 정밀하게 스캔하여 구체적인 로드맵을 작성 중입니다...


## 🚀 고수익 기업 전환 정밀 컨설팅 보고서

이 책에서 제시하는 90일 로드맵은 감성 마케팅(이모셔널 마케팅)을 활용하여 고객의 감정적인 반응을 유도하고, 이를 통해 매출을 극대화하는 전략을 중심으로 구성되어 있습니다. 각 단계에서 사장이 해야 할 행동을 다음과 같이 정리할 수 있습니다.

### 1단계: 준비 및 계획 수립 (1~2주차)
- **핵심 개념**: 이모셔널 마케팅의 이해
  - 고객의 감정적인 반응을 유도하는 마케팅 기법을 이해하고, 이를 통해 고객의 구매 결정을 촉진합니다. [p.13]
- **실행 방안**:
  - 시장 조사 및 고객 분석을 통해 타겟 고객의 감정적 니즈를 파악합니다.
  - 경쟁사의 마케팅 전략을 분석하여 차별화된 감성 마케팅 전략을 수립합니다.
- **사장의 행동**:
  - 팀과 함께 브레인스토밍 세션을 열어 고객의 감정적 니즈를 파악하고, 이를 반영한 마케팅 메시지를 개발합니다.

### 2단계: 마케팅 메시지 개발 및 테스트 (3~4주차)
- **핵심 개념**: 광고 문구 및 디자인의 중요성
  - 간결한 설명보다 충분히 설명하고, 기사풍의 레이아웃을 사용하여 고객의 관심을 끌어야 합니다. [p.182]
- **실행 방안**:
  - 다양한 광고 문구와 디자인을 테스트하여 가장 효과적인 조합을 찾습니다.
  - 고객의 피드백을 수집하여 마케팅 메시지를 지속적으로 개선합니다.
- **사장의 행동**:
  - 광고 문구와 디자인을 직접 검토하고, 고객의 반응을 분석하여 개선점을 도출합니다.

### 3단계: 고객 유치 및 관계 구축 (5~8주차)
- **핵심 개념**: 고객 획득 시스템 구축
  - 고객이 자발적으로 상품을 구입하도록 유도하는 시스템을 설계합니다. [p.188]
- **실행 방안**:
  - 고객의 구매 여정을 설계하고, 각 단계에서 필요한 마케팅 활동을 계획합니다.
  - 고객과의 지속적인 관계 구축을 위한 커뮤니케이션 전략을 수립합니다.
- **사장의 행동**:
  - 고객과의 접점을 늘리고, 고객의 피드백을 적극적으로 수용하여 관계를 강화합니다.

### 4단계: 매출 극대화 및 시스템 최적화 (9~12주차)
- **핵심 개념**: 매출과 연결되는 시스템 작동
  - 광고 선전의 반응률을 높이고, 매출과 직접 연결되는 시스템을 구축합니다. [p.188]
- **실행 방안**:
  - 마케팅 캠페인의 성과를 분석하고, 효과적인 전략을 지속적으로 최적화합니다.
  - 고객의 재구매를 유도하기 위한 프로그램을 개발합니다.
- **사장의 행동**:
  - 매출 데이터를 분석하여 성과를 평가하고, 필요한 경우 전략을 조정합니다.
  - 팀과 함께 성공 사례를 공유하고, 지속적인 개선을 위한 아이디어를 모색합니다.

이 로드맵을 통해 사장은 감성 마케팅을 활용하여 고객의 감정적 반응을 유도하고, 이를 통해 매출을 극대화하는 전략을 체계적으로 실행할 수 있습니다.

### 🚀 하이브리드 검색 (Dense + Sparse Embedding) 구현

더 정확한 결과를 위해 두 가지 검색 방식을 결합합니다:
1. **Dense Embedding (OpenAI)**: 문맥과 의미를 파악하여 검색합니다.
2. **Sparse Embedding (BM25)**: 특정 키워드(단어)의 일치 여부를 기반으로 검색합니다.

이 두 결과를 `EnsembleRetriever`로 통합하여 상호 보완적인 검색 결과를 도출합니다.

In [146]:
!pip install -q rank_bm25

In [174]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.chains import create_retrieval_chain
import importlib

# EnsembleRetriever를 찾기 위한 시도
EnsembleRetriever = None
possible_locations = [
    "langchain.retrievers.ensemble",
    "langchain_community.retrievers.ensemble",
    "langchain.retrievers",
    "langchain_community.retrievers"
]

for location in possible_locations:
    try:
        module = importlib.import_module(location)
        if hasattr(module, "EnsembleRetriever"):
            EnsembleRetriever = getattr(module, "EnsembleRetriever")
            break
    except ImportError:
        continue

if EnsembleRetriever is None:
    print("❌ EnsembleRetriever를 찾을 수 없습니다. 대안으로 BM25 검색 결과만 우선 출력합니다.")
    hybrid_retriever = db_new.as_retriever(search_kwargs={'k': 5})
else:
    # 1. Sparse Retriever (BM25) 설정
    bm25_retriever = BM25Retriever.from_documents(docs)
    bm25_retriever.k = 3

    # 2. Dense Retriever (Chroma) 설정
    chroma_retriever = db_new.as_retriever(search_kwargs={"k": 3})

    # 3. Ensemble Retriever 구축 (Hybrid Search)
    hybrid_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, chroma_retriever],
        weights=[0.5, 0.5]
    )
    print("✅ 하이브리드 검색기 구축 성공!")

# 4. 하이브리드 RAG 체인 생성
hybrid_rag_chain = create_retrieval_chain(hybrid_retriever, question_answer_chain)

# 5. 질문 실행
query_hybrid = "회사의 예금 잔액을 30일 안에 늘리기 위해 당장 실행해야 할 마케팅 액션은 무엇인가요?"
print(f"🔍 검색 실행 중: {query_hybrid}\n")

try:
    response_hybrid = hybrid_rag_chain.invoke({"input": query_hybrid})
    from IPython.display import Markdown, display
    display(Markdown(f"### 💡 마케팅 솔루션 답변\n\n{response_hybrid['answer']}"))

    print("\n[참조된 문서 출처]")
    for i, doc in enumerate(response_hybrid['context']):
        p = doc.metadata.get('page', 'N/A')
        print(f"{i+1}. 페이지: {p} | {doc.page_content[:60]}...")
except Exception as e:
    print(f"실행 중 오류 발생: {e}")

❌ EnsembleRetriever를 찾을 수 없습니다. 대안으로 BM25 검색 결과만 우선 출력합니다.
🔍 검색 실행 중: 회사의 예금 잔액을 30일 안에 늘리기 위해 당장 실행해야 할 마케팅 액션은 무엇인가요?



### 💡 마케팅 솔루션 답변

회사의 예금 잔액을 30일 안에 늘리기 위해서는 즉각적인 수익 창출을 목표로 하는 마케팅 전략이 필요합니다. 다음은 90일 계획 중 첫 30일 동안 실행할 수 있는 구체적인 액션 플랜입니다:

1. **이모셔널 마케팅 활용**: 고객의 감정을 자극하는 마케팅 메시지를 개발하세요. 고객이 제품이나 서비스에 감정적으로 연결될 수 있도록 스토리텔링을 활용합니다. 예를 들어, 고객의 문제를 해결하는 데 어떻게 도움이 되는지를 강조하는 광고 문구를 작성하세요. [p.XX]

2. **고객 세분화 및 타겟팅**: 기존 고객 데이터를 분석하여 가장 수익성이 높은 고객 세그먼트를 식별합니다. 이 세그먼트를 대상으로 맞춤형 마케팅 캠페인을 실행하여 구매 전환율을 높입니다. [p.XX]

3. **프로모션 및 할인 제공**: 단기적으로 매출을 증대시키기 위해 한정된 기간 동안 특별 할인이나 프로모션을 제공합니다. 예를 들어, 첫 구매 고객에게 10% 할인을 제공하거나, 특정 금액 이상 구매 시 무료 배송을 제공하는 등의 전략을 사용할 수 있습니다. [p.XX]

4. **고객 응대 개선**: 고객 서비스 팀을 강화하여 고객 문의에 신속하고 친절하게 대응합니다. 긍정적인 고객 경험은 재구매로 이어질 가능성이 높습니다. 고객 응대 시 고객의 이름을 부르고, 문제 해결에 적극적으로 나서는 자세를 취하세요. [p.XX]

5. **소셜 미디어 캠페인**: 소셜 미디어 플랫폼을 활용하여 브랜드 인지도를 높이고, 제품이나 서비스에 대한 관심을 유도합니다. 인플루언서와 협력하여 제품 리뷰나 추천을 받는 것도 효과적입니다. [p.XX]

이러한 전략을 통해 단기적으로 매출을 증대시키고, 예금 잔액을 늘릴 수 있습니다. 각 단계에서 고객의 반응을 모니터링하고, 필요한 경우 전략을 조정하는 것이 중요합니다. [p.XX]


[참조된 문서 출처]
1. 페이지: 3 | (좀락 풀동렉 [권량탤튼 90일만에 당신의 회사루 고수익기업으로 바뀌라 실천마켓터 간다마사노리 지음 조기선 ...
2. 페이지: 3 | (좀락 풀동렉 [권량탤튼 90일만에 당신의 회사루 고수익기업으로 바뀌라 실천마켓터 간다마사노리 지음 조기선 ...
3. 페이지: 1 | 90일만에 당신의 회사름 고수씩기업으로 바뀌라...
4. 페이지: 1 | 90일만에 당신의 회사름 고수씩기업으로 바뀌라...
5. 페이지: 17 | 그러나; 180일 후에는 2,430만엔(약2억5천만원) 회사로 친다면 대단한 금액은 아법니다: 그러나 저능 ...


In [182]:
!pip install -q flashrank

In [212]:
import importlib
from langchain_openai import ChatOpenAI
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_community.document_compressors.flashrank_rerank import FlashrankRerank
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_core.prompts import ChatPromptTemplate
from typing import List
from langchain_core.documents import Document

# 1. ContextualCompressionRetriever 로드 시도 및 Fallback 구현
# 현재 설치된 LangChain 버전에 따라 모듈 위치가 다를 수 있어 동적 로드를 시도합니다.
paths_to_try = [
    ("langchain.retrievers.contextual_compression", "ContextualCompressionRetriever"),
    ("langchain.retrievers", "ContextualCompressionRetriever")
]

ContextualCompressionRetriever = None
for module_path, class_name in paths_to_try:
    try:
        module = importlib.import_module(module_path)
        ContextualCompressionRetriever = getattr(module, class_name)
        break
    except (ImportError, AttributeError):
        continue

# 라이브러리에서 로드하지 못한 경우를 대비하여 직접 클래스를 정의합니다.
if not ContextualCompressionRetriever:
    class ContextualCompressionRetriever(BaseRetriever):
        base_compressor: object # 문서를 재정렬하는 컴프레서 (Reranker)
        base_retriever: object  # 1차 문서를 수집하는 검색기

        def _get_relevant_documents(
            self, query: str, *, run_manager: CallbackManagerForRetrieverRun = None
        ) -> List[Document]:
            # 1단계: 검색기를 통해 쿼리와 유사한 문서들을 가져옵니다.
            docs = self.base_retriever.invoke(query)
            # 2단계: 가져온 문서들을 Reranker(Flashrank)를 통해 재정렬하고 핵심 문서만 남깁니다.
            return self.base_compressor.compress_documents(docs, query)

# 2. Reranker 구성 요소 설정
# 답변 생성에 사용할 고성능 LLM 모델을 초기화합니다.
llm_rerank = ChatOpenAI(model_name="gpt-4o", temperature=0)

# FlashrankRerank: 검색된 문서들 중 가장 관련도가 높은 상위 5개만 필터링하도록 설정합니다.
compressor = FlashrankRerank(top_n=5)

# 1단계 검색기: 후보군을 충분히 확보하기 위해 Vector DB에서 20개의 문서를 먼저 가져옵니다.
base_retriever = db_new.as_retriever(search_kwargs={"k": 20})

# 검색기와 컴프레서를 결합하여 2단계 검색 시스템을 완성합니다.
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

# 3. 프롬프트 템플릿 설정
# 시스템 프롬프트: AI가 전문가 컨설턴트로서 검색된 context 내용만으로 답변하도록 지시합니다.
system_prompt = (
    "You are an expert consultant based on the provided documents. "
    "Answer using only the retrieved context.\n\n{context}"
)
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# 문서 결합 체인: 검색된 여러 문서를 하나로 합쳐 LLM에 전달하는 역할을 합니다.
question_answer_chain = create_stuff_documents_chain(llm_rerank, prompt)

# 4. 최종 Reranking RAG 체인 조립
# 검색기와 답변 생성 체인을 연결하여 최종 파이프라인을 구축합니다.
rerank_rag_chain = create_retrieval_chain(compression_retriever, question_answer_chain)

print("✅ Reranker 기반의 2단계 검색 체인이 복구 및 재설정되었습니다.")

✅ Reranker 기반의 2단계 검색 체인이 복구 및 재설정되었습니다.


In [213]:
# 질문 설정: 고수익 전환을 위한 습관과 마인드셋에 대해 질문합니다.
query_rerank = "회사의 고수익 전환을 위해 사장이 당장 버려야 할 습관과 갖춰야 할 마인드셋은 무엇인가요?"

print(f"🔍 Reranker를 통한 검색 및 답변 생성 시작: {query_rerank}\n")

try:
    # 2단계 검색(Reranking) 파이프라인 호출
    # 1단계에서 20개를 검색하고, Flashrank를 통해 가장 관련성이 높은 5개를 선별하여 답변을 생성합니다.
    response_rerank = rerank_rag_chain.invoke({"input": query_rerank})

    from IPython.display import Markdown, display
    # LLM이 생성한 최종 답변을 마크다운 형식으로 출력합니다.
    display(Markdown(f"### 💡 Reranker 기반 분석 답변\n\n{response_rerank['answer']}"))

    print("\n[Reranking을 통해 선별된 핵심 근거 문서]")
    # 답변 생성에 실제로 사용된 상위 5개의 문서 출처와 내용을 확인합니다.
    for i, doc in enumerate(response_rerank['context']):
        page_num = doc.metadata.get('page', 'N/A')
        content_preview = doc.page_content[:100].replace('\n', ' ')
        print(f"{i+1}. 페이지: {page_num} | 내용: {content_preview}...")

except Exception as e:
    # 실행 중 오류가 발생할 경우 에러 메시지를 출력합니다.
    print(f"❌ 실행 중 오류 발생: {e}")

🔍 Reranker를 통한 검색 및 답변 생성 시작: 회사의 고수익 전환을 위해 사장이 당장 버려야 할 습관과 갖춰야 할 마인드셋은 무엇인가요?



### 💡 Reranker 기반 분석 답변

회사의 고수익 전환을 위해 사장이 당장 버려야 할 습관은 고객의 이동을 어렵게 만드는 경사를 높이는 것입니다. 대신, 고객이 쉽게 이동할 수 있도록 다음 계단을 만들어 주는 것이 중요합니다. 또한, 판매 방법의 연구를 철저히 하여 정직하게 좋은 상품을 제공하는 것이 필요합니다. 

사장이 갖춰야 할 마인드셋은 필요 이상의 영업을 줄이고, 효율적인 방법으로 고객을 유치하는 것입니다. 이를 통해 영업 경비를 줄이고, 매출을 성장시키는 방향으로 나아가야 합니다.


[Reranking을 통해 선별된 핵심 근거 문서]
1. 페이지: 199 | 내용: 켜이스 1 법인올 상대로 한 영업으로 신규고객올 개척하는 경우 내구소비재(대리점 판매) 서비스업(직접 판매) 테이스 2 소비자흘 상대로 한 영업으로 신규고객올 개척하는 경우 2-1...
2. 페이지: 196 | 내용: 경비가 발생하지 않듣다는 것은 아법니다만 고 객이 이동하기 쉽제 다음 계단올 만들어 준다라는 것입니다. 대 부분의 회사는 이 때의 경사가 너무 높아서 필요이상의 영업올 19 용일만...
3. 페이지: 3 | 내용: (좀락 풀동렉 [권량탤튼 90일만에 당신의 회사루 고수익기업으로 바뀌라 실천마켓터 간다마사노리 지음 조기선 옮김 마신시립합포도서관 I대 배 EM0OO0037932 중앙이아이피...
4. 페이지: 3 | 내용: (좀락 풀동렉 [권량탤튼 90일만에 당신의 회사루 고수익기업으로 바뀌라 실천마켓터 간다마사노리 지음 조기선 옮김 마신시립합포도서관 I대 배 EM0OO0037932 중앙이아이피...
5. 페이지: 78 | 내용: 커버하지 않으면 안되니다: 그래서 판매방법의 연구름 철저히 해앗습니다: 이에 비 해, 정직한 사람은 어설프게 좋은 상품올 갖고 있습니다: 그렇기 90일만에 당신의 회사흘 고수의기업...
